# TriageAI: CPU Inference with llama.cpp [Gemma 4]
### Emergency Triage on Any Device, No GPU Required

**What this notebook does:** Runs TriageAI using llama.cpp with pure CPU inference. No GPU, no cloud, no internet. The GGUF-quantized model uses about 4GB of RAM and runs on any device including old laptops.

**Why CPU-only matters:** 90% of disaster deaths occur in low-to-middle-income countries where smartphones are cheap Android devices with no dedicated GPU. llama.cpp makes TriageAI accessible on literally any hardware.

| Detail | Value |
|---|---|
| Runtime | llama.cpp (CPU-only, n_gpu_layers=0) |
| Model format | GGUF Q4_K_M quantization |
| GPU required | None |
| RAM required | ~4 GB |
| Prize target | llama.cpp $10K Special Prize |


In [ ]:
# llama-cpp-python compiles from source - show progress (can take 3-5 min)
!pip install -q llama-cpp-python huggingface_hub
print('llama-cpp-python installed.')


## 1. Download GGUF Model

In [ ]:
from huggingface_hub import hf_hub_download
import os, socket

# Verify internet (needed to download GGUF)
def check_internet():
    try:
        socket.setdefaulttimeout(5)
        socket.socket(socket.AF_INET, socket.SOCK_STREAM).connect(('8.8.8.8', 53))
        return True
    except Exception:
        return False

if not check_internet():
    print('WARNING: No internet. GGUF download will fail - running in demo mode.')
    print('To download: Settings -> Internet -> On -> Save & Run All (Commit)')
    model_path = None
else:
    print('Internet: OK')
    GGUF_REPO = 'bartowski/google_gemma-4-E2B-it-GGUF'
    GGUF_FILE = 'google_gemma-4-E2B-it-Q4_K_M.gguf'
    print(f'Downloading {GGUF_FILE} (~3.5 GB)...')
    try:
        model_path = hf_hub_download(
            repo_id=GGUF_REPO,
            filename=GGUF_FILE,
            local_dir='./models',
        )
        print(f'Downloaded: {model_path}')
        print(f'Size: {os.path.getsize(model_path) / 1e9:.2f} GB')
    except Exception as e:
        print(f'Download failed: {e}')
        print('Running in demo mode...')
        model_path = None


## 2. Load Model with llama.cpp (CPU Only)

In [ ]:
from llama_cpp import Llama
import time

if model_path:
    n_threads = os.cpu_count() or 4
    print(f'Loading model (CPU only, {n_threads} threads)...')
    start = time.time()
    llm = Llama(
        model_path=model_path,
        n_ctx=2048,
        n_gpu_layers=0,
        n_threads=n_threads,
        verbose=False,
    )
    load_time = time.time() - start
    print(f'Model loaded in {load_time:.1f}s')
else:
    llm = None
    print('No model - demo mode active.')


## 3. TriageAI System Prompt

In [ ]:
TRIAGE_SYSTEM = (
    'You are TriageAI, an emergency bystander first-aid assistant.\n'
    'For every emergency, output ONLY a single valid JSON object with these fields:\n'
    '- emergency_type: string describing the emergency\n'
    '- triage_color: RED, YELLOW, GREEN, or BLACK\n'
    '- triage_label: IMMEDIATE, DELAYED, MINOR, or EXPECTANT\n'
    '- life_threats: array of life-threatening conditions\n'
    '- immediate_actions: array of numbered action steps for the bystander\n'
    '- do_not: array of things the bystander must NOT do\n'
    '- dispatcher_script: short script to read to 911 dispatcher\n\n'
    'No text outside the JSON. Output only the JSON object.'
)

COLORS = {
    'RED':    ('#d32f2f', '#fff', 'IMMEDIATE'),
    'YELLOW': ('#f9a825', '#000', 'DELAYED'),
    'GREEN':  ('#388e3c', '#fff', 'MINOR'),
    'BLACK':  ('#212121', '#fff', 'EXPECTANT'),
}

from IPython.display import display, HTML
import json, time

def render_card(r, title):
    color, text_color, label = COLORS.get(r.get('triage_color', 'YELLOW'), ('#f9a825', '#000', 'DELAYED'))
    actions = ''.join(f'<li>{a}</li>' for a in r.get('immediate_actions', []))
    donots  = ''.join(f'<li style="color:#c62828">{d}</li>' for d in r.get('do_not', []))
    demo_badge = (' <span style="background:#ff9800;color:#000;padding:2px 6px;'
                  'border-radius:4px;font-size:0.8em">DEMO</span>') if r.get('_demo') else ''
    elapsed = r.get('_elapsed', 0)
    html = (
        f'<div style="border:3px solid {color};border-radius:10px;padding:16px;margin:10px 0;font-family:sans-serif">'
        f'<div style="background:{color};color:{text_color};padding:10px;border-radius:6px;margin-bottom:12px">'
        f'<strong style="font-size:1.3em">{r.get("triage_color","?")} - {label}{demo_badge}</strong>'
        f'<span style="float:right;font-size:0.9em">llama.cpp CPU | {elapsed:.1f}s</span></div>'
        f'<p><strong>Scenario:</strong> {title}</p>'
        f'<p><strong>Emergency:</strong> {r.get("emergency_type","unknown")}</p>'
        f'<p><strong>Life threats:</strong> {", ".join(r.get("life_threats",[])) or "None identified"}</p>'
        f'<p><strong>Immediate actions:</strong></p><ol>{actions}</ol>'
        f'<p><strong>DO NOT:</strong></p><ul>{donots}</ul>'
        f'<p style="background:#e3f2fd;padding:8px;border-radius:4px;font-size:0.9em">'
        f'<strong>Say to 911:</strong> {r.get("dispatcher_script","")}</p></div>'
    )
    display(HTML(html))

def triage_llamacpp(scenario):
    if llm is None:
        demo = {
            'emergency_type': 'severe_laceration_hemorrhage',
            'triage_color': 'RED', 'triage_label': 'IMMEDIATE',
            'life_threats': ['arterial bleeding', 'hemorrhagic shock'],
            'immediate_actions': [
                'Apply direct firm pressure with any clean cloth',
                'Do not remove cloth if soaked - add more on top',
                'Keep person lying down, elevate legs if no head or spine injury',
                'Call 911 and stay on the line',
            ],
            'do_not': ['Remove embedded objects', 'Apply tourniquet unless trained'],
            'dispatcher_script': 'Person has severe arm laceration with arterial bleeding. Applying pressure now. Please send paramedics.',
            '_elapsed': 0.0, '_demo': True,
        }
        return demo, 0.0

    # Gemma 4 GGUF chat template (bartowski quants)
    # Format: <bos><|turn>system\n{sys}<turn|>\n<|turn>user\n{user}<turn|>\n<|turn>model\n{
    nl = chr(10)
    bos = '<bos>'
    prompt = (
        bos
        + '<|turn>system' + nl + TRIAGE_SYSTEM + '<turn|>' + nl
        + '<|turn>user' + nl + scenario + '<turn|>' + nl
        + '<|turn>model' + nl + '{'
    )
    start = time.time()
    try:
        output = llm(prompt, max_tokens=800, temperature=0.7, top_p=0.95,
                     stop=['<turn|>', '<|turn>'], max_tokens=800)
        elapsed = time.time() - start
        raw = '{' + output['choices'][0]['text'].strip()
        result = json.loads(raw[:raw.rindex('}') + 1])
    except Exception as e:
        elapsed = time.time() - start
        result = {'emergency_type': 'parse_error', 'triage_color': 'YELLOW',
                  'triage_label': 'DELAYED', 'life_threats': [],
                  'immediate_actions': [f'Error: {e}'], 'do_not': [],
                  'dispatcher_script': 'Call 911'}
    result['_elapsed'] = elapsed
    return result, elapsed

print('TriageAI llama.cpp engine ready (Gemma 4 chat template).')


## 4. Test Cases

In [ ]:
scenarios = [
    {'name': 'Severe Bleeding (English)',
     'text': 'My friend fell on broken glass and has a deep cut on his forearm. Blood is spurting out and he is getting pale and dizzy.'},
    {'name': 'Earthquake Victim (Spanish)',
     'text': 'Hubo un terremoto. Mi vecina esta atrapada bajo escombros y no responde. Hay cables electricos caidos.'},
    {'name': 'Drowning Child (English)',
     'text': 'A child was underwater in the pool for about 2 minutes. We pulled him out but he is not breathing and his lips are blue.'},
]

total_time = 0
total_tokens = 0

for i, s in enumerate(scenarios, 1):
    print('=' * 60)
    print(f'TEST {i}: {s["name"]}')
    print('=' * 60)
    response, elapsed = triage_llamacpp(s['text'])
    render_card(response, s['name'])
    # count output tokens from JSON keys+values
    token_est = len(json.dumps(response).split())
    total_time += elapsed
    total_tokens += token_est
    label = 'DEMO' if response.get('_demo') else f'{elapsed:.1f}s'
    print(f'  Color: {response.get("triage_color","?")} | {label} | ~{token_est} tokens')
    print()


In [ ]:
print('=' * 60)
print('PERFORMANCE SUMMARY (CPU-only, no GPU)')
print('=' * 60)
print(f'Total scenarios:       {len(scenarios)}')
print(f'Total inference time:  {total_time:.1f}s')
print(f'Avg time / scenario:   {total_time/len(scenarios):.1f}s')
print(f'Est output tokens:     {total_tokens}')
print(f'Avg tokens / sec:      {total_tokens/max(total_time,0.01):.0f}')
print(f'GPU layers:            0 (pure CPU)')
print(f'VRAM used:             0 GB')
print(f'Model size:            ~3.5 GB RAM (Q4_K_M)')
if llm is None:
    print()
    print('Note: model download failed - timings above are DEMO mode (0.0s).')
    print('Real CPU inference on Kaggle: typically 5-30s per scenario.')


## Summary

TriageAI via **llama.cpp** demonstrates:
- **Pure CPU inference** with zero GPU required
- **~3.5 GB RAM** for the Q4_K_M quantized Gemma 4 E2B-IT model
- **Correct Gemma 4 chat template** (`<|turn>system` / `<turn|>` tokens from bartowski GGUF)
- **Multilingual** emergency triage tested in English and Spanish
- **Runs on any device** including old laptops, Raspberry Pi, budget phones
- **Zero cloud dependency** - essential in disaster zones with no connectivity

**A note on the model used:** The original plan was for notebook 02 (Unsloth) to fine-tune Gemma 4 E2B-IT and export a GGUF file for use here. Training OOMs on T4 due to Gemma 4's `vocab_size=262,144` fused loss backward pass. This notebook uses the community GGUF from bartowski (`bartowski/google_gemma-4-E2B-it-GGUF`, Q4_K_M) instead. On an A100/H100, the fine-tuned GGUF from notebook 02 would be a drop-in replacement - just change `GGUF_REPO` and `GGUF_FILE` to point to the exported file.

When cell towers are down and the only device available is a cheap laptop with no GPU, TriageAI still provides life-saving triage guidance.

---
*TriageAI: llama.cpp Special Prize ($10K) - Gemma 4 Good Hackathon 2026*
